In [ ]:
import sys
from pathlib import Path

ROOT = Path("..")
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import csv
from collections import defaultdict

from lob import LimitOrderBook, Order, Side
from lob_reader import LOBReader, Snapshot

# Avellaneda-Stoikov Market Making Strategy

## 2008


## 1. Core Idea (Skeptical Summary)

The market maker maximizes expected exponential utility of terminal wealth including inventory valued at the terminal mid-price. The model accounts for:
- Inventory risk (adverse price moves while holding q ≠ 0).
- Adverse selection (informed traders hitting your quotes).
- Stochastic order arrivals.

The solution yields a **reservation price** (indifference price) around which the MM quotes symmetrically. The reservation price shifts opposite to inventory to induce mean-reversion. The optimal spread balances risk aversion, volatility, time-to-horizon, and market liquidity.

This is an asymptotic closed-form approximation under exponential arrival intensities and small-inventory linearization.

## 2. Model Assumptions

- Mid-price follows driftless Brownian motion: $ dS_t = \sigma \, dW_t $

- Market orders arrive as independent Poisson processes with intensity:  
  $ \lambda(\delta) = A \exp(-\kappa \delta) $ 
  where $\delta$ is the distance of the quote from the current mid-price.  
  $\kappa > 0$ controls how fast fill probability decays with worse pricing.

- Finite trading horizon [0, T].  
- Exponential utility with risk aversion $\gamma > 0$: $ U(W) = -\exp(-\gamma W) $.  
- No transaction costs, continuous quote updating, no market impact from own orders.

## 3. Required Parameters and Data

### Fixed (calibrated once or tuned)
- $\gamma$: Risk aversion parameter (higher → more conservative, stronger inventory control). Typical range: 0.1 – 1.0.
- $\kappa$: Liquidity parameter (inverse sensitivity of arrival rate to distance). Calibrated from order book or historical fill data. Higher $\kappa$ → deeper book → tighter spreads.
- $\sigma$: Annualized or period volatility of mid-price (must be in consistent time units with T-t).
- T: Terminal time (normalize to 1.0 for fraction of session, or use seconds/hours consistently).
- A: Maximum arrival intensity at $\delta=0$ (used for calibration of $\kappa$, often not needed for quoting).

### Dynamic (updated every tick or regularly)
- $ s $: Current mid-price ($ s = (best\_bid + best\_ask)/2 $).
- $ q $: Current inventory in base asset (positive = long, negative = short, integer or fractional depending on asset).
- $ t $: Current time (fraction of T or absolute; compute remaining time $\tau = T - t$).
- Realized volatility estimate (rolling window on mid-price returns to update $\sigma$).

**Data sources needed for live trading:**
- Real-time Level-1 or Level-2 quotes (for mid-price and optionally order-book shape for $\kappa$ calibration).
- Own position tracking (inventory q and cash).
- Historical mid-price time series for volatility estimation.
- Fill notifications to update q immediately.

## 4. Core Formulas

### 4.1 Reservation Price (Indifference Price)
$$
r(s, q, \tau) = s - q \cdot \gamma \cdot \sigma^2 \cdot \tau
$$
where $\tau = T - t$.

This is the price at which the MM is indifferent to holding the current inventory q. Positive q pushes r below s (encourages selling), negative q pushes r above s (encourages buying).

### 4.2 Optimal Spread
The symmetric half-spread $\frac{\delta^*}{2}$ (or total optimal spread $\delta^*$) is:
$$
\delta^* = \gamma \cdot \sigma^2 \cdot \tau + \frac{2}{\gamma} \ln\left(1 + \frac{\gamma}{\kappa}\right)
$$
The term $\frac{2}{\gamma} \ln(1 + \gamma/\kappa)$ is the liquidity/adverse-selection component (approximately constant). The first term is the inventory-risk component that grows with time and volatility.

### 4.3 Optimal Bid and Ask Quotes
$$
bid = r - \frac{\delta^*}{2} = s - q \cdot \gamma \cdot \sigma^2 \cdot \tau - \frac{\delta^*}{2}
$$
$$
ask = r + \frac{\delta^*}{2} = s - q \cdot \gamma \cdot \sigma^2 \cdot \tau + \frac{\delta^*}{2}
$$

In code these are the limit prices at which you post buy (bid) and sell (ask) orders.

## 5. What the Strategy Achieves

- Automatic inventory mean-reversion without explicit target.
- Spread that widens with volatility and time remaining.
- Quotes that are optimal under the model's assumptions.
- Closed-form, computationally cheap – suitable for high-frequency execution.

In [ ]:
data_dir = ROOT / "data" / "MD_small"

In [ ]:
def _parse_ts_us(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s.astype("int64"), unit="us")


# ── LOB snapshots via Snapshot.from_row() ─────────────────────────────────────
snapshots: list[Snapshot] = []
with open(data_dir / "lob.csv", newline="") as fh:
    for row in csv.DictReader(fh):
        snapshots.append(Snapshot.from_row(row))


# Derive lob_df from snapshots (best bid/ask → mid)
def _best_bid(snap: Snapshot) -> float:
    return max(snap.bids.keys()) if snap.bids else float("nan")


def _best_ask(snap: Snapshot) -> float:
    return min(snap.asks.keys()) if snap.asks else float("nan")


lob_df = pd.DataFrame(
    {
        "ts": pd.to_datetime([s.timestamp for s in snapshots], unit="us"),
        "bid": [_best_bid(s) for s in snapshots],
        "ask": [_best_ask(s) for s in snapshots],
    }
)
lob_df["mid"] = 0.5 * (lob_df["bid"] + lob_df["ask"])
lob_df["dt"] = lob_df["ts"].diff().dt.total_seconds().fillna(0.0).clip(lower=0.0)
lob_df = lob_df.sort_values("ts").reset_index(drop=True)

# ── Trades ─────────────────────────────────────────────────────────────────────
tr_raw = pd.read_csv(data_dir / "trades.csv", index_col=0)
trades_df = pd.DataFrame(
    {
        "ts": _parse_ts_us(tr_raw["local_timestamp"]),
        "side_n": tr_raw["side"]
        .str.strip()
        .str.lower()
        .map({"buy": 1, "sell": -1})
        .astype("int8"),
        "price": tr_raw["price"].astype(float),
        "amount": tr_raw["amount"].astype(float),
    }
)
trades_df = trades_df.dropna().sort_values("ts").reset_index(drop=True)

print(f"Snapshots loaded: {len(snapshots)}")
print("LOB dataframe:", lob_df.shape)
display(lob_df[["ts", "bid", "ask", "mid"]].head())
print("\nTrades:", trades_df.shape)
display(trades_df.head())

## Calibration

In [ ]:
# ── σ: realized volatility (units: price / sqrt(second)) ──────────────────────
dm = lob_df["mid"].diff().to_numpy()
dt = lob_df["dt"].to_numpy()
valid = np.isfinite(dm) & (dt > 0)
sigma = float(np.sqrt(np.mean((dm[valid] ** 2) / dt[valid])))

# ── tick size ──────────────────────────────────────────────────────────────────
prices = np.concatenate([lob_df["bid"].to_numpy(), lob_df["ask"].to_numpy()])
diffs = np.diff(np.unique(np.round(prices, 10)))
tick = float(np.quantile(diffs[diffs > 0], 0.05))

# ── κ: fit λ(δ) = A·exp(−κ·δ) via OLS on log-histogram ───────────────────────
# Match each trade to the most recent LOB mid
tr_ts_ns = trades_df["ts"].values.astype("int64")
lob_ts_ns = lob_df["ts"].values.astype("int64")
idx = np.searchsorted(lob_ts_ns, tr_ts_ns, side="right") - 1
idx = np.clip(idx, 0, len(lob_df) - 1)

mid_at_trade = lob_df["mid"].to_numpy()[idx]
delta = np.where(
    trades_df["side_n"].to_numpy() > 0,
    trades_df["price"].to_numpy() - mid_at_trade,  # buy: how far above mid
    mid_at_trade - trades_df["price"].to_numpy(),  # sell: how far below mid
)
delta = delta[np.isfinite(delta) & (delta >= 0)]

T_session = (lob_df["ts"].iloc[-1] - lob_df["ts"].iloc[0]).total_seconds()
bins = np.linspace(0.0, float(np.quantile(delta, 0.99)), 25)
cnt, edges = np.histogram(delta, bins=bins)
centers = 0.5 * (edges[:-1] + edges[1:])
rates = cnt / T_session

mask = rates > 0
x, y = centers[mask], np.log(rates[mask])
X = np.column_stack([np.ones(len(x)), x])
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
A_calib = float(np.exp(beta[0]))
kappa = float(max(-beta[1], 1e-6))

print(f"sigma  = {sigma:.8f}  (price / √s)")
print(f"tick   = {tick:.8f}")
print(f"A      = {A_calib:.6f}  (arrival rate at δ=0, per second)")
print(f"κ      = {kappa:.4f}  (decay of arrival rate with distance)")
print(f"λ(0)   = {A_calib:.4f},  λ(tick) = {A_calib * np.exp(-kappa * tick):.4f}")

## AS(2008) Quoting

In [ ]:
def quote_as2008(
    mid: float,
    q: float,
    tau: float,
    gamma: float,
    sigma: float,
    kappa: float,
    tick: float,
) -> tuple[float, float]:
    """
    AS(2008) reservation price + symmetric optimal spread.
    """
    sigma2 = sigma * sigma
    r = mid - q * gamma * sigma2 * tau
    delta = (1.0 / gamma) * np.log(
        1.0 + gamma / max(kappa, 1e-9)
    ) + 0.5 * gamma * sigma2 * tau
    bid = np.floor((r - delta) / tick) * tick
    ask = np.ceil((r + delta) / tick) * tick
    if ask <= bid:
        ask = bid + tick
    return bid, ask

## Backtest

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
GAMMA = 0.05
T_SECONDS = 3600.0
ORDER_SIZE = 10_000.0
Q_MAX = 500_000.0

# ── Prepare trade lookup by LOB interval ──────────────────────────────────────
lob_ts_ns = lob_df["ts"].values.astype("int64")
trade_ts_ns = trades_df["ts"].values.astype("int64")
trade_interval = np.searchsorted(lob_ts_ns, trade_ts_ns, side="right") - 1


interval_to_trades: dict[int, list[int]] = defaultdict(list)
for trade_row, lob_idx in enumerate(trade_interval):
    if 0 <= lob_idx < len(lob_df) - 1:
        interval_to_trades[int(lob_idx)].append(trade_row)

# ── Two-LOB simulation ─────────────────────────────────────────────────────────
market_lob = LimitOrderBook()
mm_lob = LimitOrderBook()
reader = LOBReader(market_lob)

t0 = lob_df["ts"].iloc[0]
inv = 0.0
cash = 0.0
bid_oid = None
ask_oid = None
bid_q = None
ask_q = None

rows = []

for snap_idx, snap in enumerate(snapshots):
    reader.ingest(snap)  # reconcile market_lob with this snapshot
    mid = market_lob.mid_price()
    if mid is None:
        continue

    ts = lob_df["ts"].iloc[snap_idx]
    elapsed = (ts - t0).total_seconds()
    tau = max(T_SECONDS - elapsed, 0.0)

    # ── 1. Cancel previous MM quotes ─────────────────────────────────────────
    if bid_oid and mm_lob.order_exists(bid_oid):
        mm_lob.cancel_order(bid_oid)
    if ask_oid and mm_lob.order_exists(ask_oid):
        mm_lob.cancel_order(ask_oid)

    # ── 2. Compute and place new AS(2008) quotes ──────────────────────────────
    bid_q, ask_q = quote_as2008(mid, inv, tau, GAMMA, sigma, kappa, tick)

    if inv + ORDER_SIZE <= Q_MAX:
        bid_order = Order(elapsed, bid_q, ORDER_SIZE, Side.BID)
        mm_lob.place_order(bid_order)
        bid_oid = bid_order.order_id
    else:
        bid_oid = None

    if inv - ORDER_SIZE >= -Q_MAX:
        ask_order = Order(elapsed, ask_q, ORDER_SIZE, Side.ASK)
        mm_lob.place_order(ask_order)
        ask_oid = ask_order.order_id
    else:
        ask_oid = None

    # ── 3. Process trades in this interval as IOC orders on mm_lob ───────────
    buy_fill_total = 0.0
    sell_fill_total = 0.0

    for tr_row in interval_to_trades.get(snap_idx, []):
        tr = trades_df.iloc[tr_row]
        tr_price = float(tr["price"])
        tr_amount = float(tr["amount"])

        if tr["side_n"] > 0:
            # Market BUY → IOC bid → can lift MM's resting ask
            ioc = Order(elapsed, tr_price, tr_amount, Side.BID)
            fills = mm_lob.place_order(ioc)
            if mm_lob.order_exists(ioc.order_id):
                mm_lob.cancel_order(ioc.order_id)
            for f in fills:
                if f.ask_order_id == ask_oid:
                    inv -= f.fill_size
                    cash += f.fill_size * f.fill_price
                    sell_fill_total += f.fill_size
        else:
            # Market SELL → IOC ask → can hit MM's resting bid
            ioc = Order(elapsed, tr_price, tr_amount, Side.ASK)
            fills = mm_lob.place_order(ioc)
            if mm_lob.order_exists(ioc.order_id):
                mm_lob.cancel_order(ioc.order_id)
            for f in fills:
                if f.bid_order_id == bid_oid:
                    inv += f.fill_size
                    cash -= f.fill_size * f.fill_price
                    buy_fill_total += f.fill_size

    rows.append(
        {
            "ts": ts,
            "elapsed": elapsed,
            "mid": mid,
            "bid_q": bid_q,
            "ask_q": ask_q,
            "buy_fill": buy_fill_total,
            "sell_fill": sell_fill_total,
            "inventory": inv,
            "cash": cash,
            "equity": cash + inv * mid,
        }
    )

result = pd.DataFrame(rows)
print("Backtest complete:", result.shape)
display(result[["ts", "mid", "bid_q", "ask_q", "inventory", "equity"]].head(10))

## Summary Metrics

In [ ]:
eq = result["equity"].to_numpy()
deq = np.diff(eq, prepend=eq[0])
std = np.std(deq)
sharpe = float(np.mean(deq) / std) if std > 0 else 0.0

total_turnover = float(result["buy_fill"].sum() + result["sell_fill"].sum())

metrics = pd.DataFrame(
    [
        {
            "Final PnL (equity)": f"{eq[-1]:+.6f}",
            "Final inventory": f"{result['inventory'].iloc[-1]:+.0f}",
            "Avg |inventory|": f"{result['inventory'].abs().mean():.0f}",
            "Buy fills (intervals)": f"{(result['buy_fill'] > 0).sum()}",
            "Sell fills (intervals)": f"{(result['sell_fill'] > 0).sum()}",
            "Total turnover (units)": f"{total_turnover:,.0f}",
            "Sharpe-like (step)": f"{sharpe:.6f}",
        }
    ]
).T.rename(columns={0: "value"})

display(metrics)

## Plots

In [ ]:
t = result["elapsed"].to_numpy()
mid = result["mid"].to_numpy()
bq = result["bid_q"].to_numpy()
aq = result["ask_q"].to_numpy()
inv = result["inventory"].to_numpy()
eq = result["equity"].to_numpy()
cum_turnover = np.cumsum(result["buy_fill"].to_numpy() + result["sell_fill"].to_numpy())

fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)

# ── Panel 1: mid-price and MM quotes ─────────────────────────────────────────
ax = axes[0]
ax.plot(t, mid, color="#888888", linewidth=0.8, label="mid")
ax.plot(
    t, bq, color="#2ca85a", linewidth=0.7, linestyle="--", alpha=0.85, label="bid quote"
)
ax.plot(
    t, aq, color="#e04545", linewidth=0.7, linestyle="--", alpha=0.85, label="ask quote"
)
ax.set_ylabel("price")
ax.set_title("AS(2008) Market Making — MD_small", fontsize=11)
ax.legend(fontsize=8, loc="upper left")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.6f"))

# ── Panel 2: equity / PnL ─────────────────────────────────────────────────────
ax = axes[1]
ax.plot(t, eq, color="steelblue", linewidth=0.9)
ax.axhline(0, color="#aaa", linewidth=0.6, linestyle=":")
ax.set_ylabel("equity (PnL)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.4f"))

# ── Panel 3: inventory ────────────────────────────────────────────────────────
ax = axes[2]
ax.fill_between(t, inv, 0, where=(inv >= 0), color="#2ca85a", alpha=0.35, label="long")
ax.fill_between(t, inv, 0, where=(inv < 0), color="#e04545", alpha=0.35, label="short")
ax.plot(t, inv, color="#555555", linewidth=0.6)
ax.axhline(0, color="#aaa", linewidth=0.6, linestyle=":")
ax.set_ylabel("inventory (units)")
ax.legend(fontsize=8, loc="upper left")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# ── Panel 4: cumulative turnover ──────────────────────────────────────────────
ax = axes[3]
ax.plot(t, cum_turnover, color="darkorange", linewidth=0.9)
ax.set_ylabel("cumulative turnover")
ax.set_xlabel("elapsed time (s)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.show()